# 07 - Evaluasi Kwitansi Extractor (VLM LightOnOCR-2-1B) di Google Colab

Notebook ini **khusus dijalankan di Google Colab dengan GPU** (pola sama
seperti `05_test_report_agent_colab.ipynb`) — mengukur akurasi
`utils/kwitansi_extractor.py` (model VLM `LightOnOCR-2-1B`) terhadap 96
kwitansi berlabel di `data/kwitansi/ground_truth_susantojaya_2tahun.csv`
+ `data/kwitansi/kwitansi_images/`.

Yang diukur:
1. **Field-level accuracy** — exact-match rate per kolom hasil parse
   (`jenis_kwitansi`, `no_kwitansi`, `tanggal`, `pihak_terkait`,
   `nama_usaha`, `nik_pemilik`, `total`) vs ground truth.
2. **Row-success rate** — proporsi kwitansi yang SEMUA kolomnya benar
   sekaligus (bukan rata-rata per-kolom) — representasi realistis dari
   "berapa kwitansi yang bisa dipakai TANPA koreksi manual".
3. **Dampak ke `compute_monthly_estimates()`** — apakah
   `monthly_turnover_est`/`transaction_frequency_monthly` hasil hitung dari
   ekstraksi OCR mendekati angka "asli" (dihitung dari `total_amount`
   ground truth lewat FUNGSI PRODUKSI YANG SAMA, bukan reimplementasi
   terpisah — supaya metodologinya apple-to-apple).

`extract_zip_bytes()` (jalur produksi utama — zip 96 gambar in-memory,
simulasi upload user) dipakai untuk evaluasi utama; `extract_uploaded_files()`
divalidasi terpisah di sample kecil untuk memastikan kedua entry point
publik menghasilkan hasil identik (keduanya memanggil `_extract_paths()`
internal yang sama) — supaya tidak perlu menjalankan inferensi 96 gambar
dua kali.

> ⚠️ **Notebook ini belum pernah dieksekusi** — dibuat di environment
> tanpa GPU. Jalankan dari atas ke bawah di Colab.

## Sebelum mulai

1. **Ganti runtime ke GPU**: menu `Runtime` → `Change runtime type` →
   pilih `T4 GPU` (gratis) atau lebih tinggi kalau tersedia.
2. Pastikan perubahan terbaru project sudah ke-push ke GitHub (`git push`)
   sebelum menjalankan notebook ini — sel clone di bawah mengambil dari
   `origin/main`.
3. `lightonai/LightOnOCR-2-1B` adalah model publik (tidak digated seperti
   Gemma) — tidak perlu login HuggingFace.

In [ ]:
!nvidia-smi

## 1. Clone Repo & Install Dependencies

`torch`/`transformers`/`pillow` sengaja **tidak** ada di `requirements.txt`
project (itu khusus untuk deploy Streamlit Cloud yang tidak punya GPU) —
jadi diinstall terpisah di sini. `pillow` di-pin ke versi tertentu (bukan
`-U` ke terbaru) supaya tidak bentrok dengan `torchvision` bawaan Colab —
lihat komentar di sel install.

In [ ]:
REPO_URL = "https://github.com/indahsyafhyra12/Capstone-Project-ODP-DA-Asek.git"
REPO_DIR = "Capstone-Project-ODP-DA-Asek"

import os
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}
%cd {REPO_DIR}
!git pull

In [ ]:
!pip install -q -r requirements.txt
!pip install -q -U transformers accelerate huggingface_hub
# Pillow SENGAJA di-pin (bukan -U ke terbaru) - torchvision bawaan Colab
# di-build terhadap ABI Pillow versi lama; Pillow terbaru mengubah/hapus
# PIL._typing._Ink yang dipakai torchvision, menyebabkan:
#   ImportError: cannot import name '_Ink' from 'PIL._typing'
# 9.5.0 dikonfirmasi kompatibel dgn torchvision bawaan Colab (Image.open/
# .convert dipakai kwitansi_extractor.py tidak butuh fitur Pillow terbaru).
!pip install -q "pillow==9.5.0"

## 2. Import Modul Project & Muat Ground Truth

Dijalankan dari root repo (`%cd` di atas), jadi `utils.*` bisa langsung
di-import tanpa perlu `sys.path` tambahan.

In [ ]:
import io
import time
import zipfile
from pathlib import Path

import pandas as pd

from utils.kwitansi_extractor import (
    extract_zip_bytes, extract_uploaded_files, compute_monthly_estimates,
    build_raw_text_export, DETAIL_COLUMNS, MODEL_ID,
)

print(f"Model target: {MODEL_ID}")

GT_PATH = Path("data/kwitansi/ground_truth_susantojaya_2tahun.csv")
IMG_DIR = Path("data/kwitansi/kwitansi_images")

ground_truth = pd.read_csv(GT_PATH, dtype={"nik": str})
image_files = sorted(IMG_DIR.glob("*.jpg"))
print(f"Ground truth: {len(ground_truth)} baris | Gambar ditemukan: {len(image_files)}")
assert len(ground_truth) == len(image_files), "Jumlah ground truth dan gambar tidak sama - cek folder/CSV."
assert set(ground_truth["file_name"]) == {p.name for p in image_files}, "Nama file di ground truth tidak persis sama dgn folder gambar."

## 3. Preload Model (Sekali Saja)

Panggilan pertama yang men-trigger download + load model ke GPU (bisa
beberapa menit tergantung koneksi) — dijalankan di sel terpisah supaya
waktu download tidak tercampur ke timing ekstraksi di sel berikutnya.

In [ ]:
_warm_start = time.perf_counter()
from utils.kwitansi_extractor import _load_model
_model, _processor, _device, _dtype = _load_model()
print(f"Model dimuat dalam {time.perf_counter() - _warm_start:.1f}s")
print(f"Device: {_device}, dtype: {_dtype}")

## 4. Jalankan Ekstraksi (`extract_zip_bytes()` — jalur produksi utama)

Zip 96 gambar in-memory (simulasi upload ZIP seperti di halaman
"Pengajuan Credit Baru"/"Laporan Keuangan Kwitansi"), lalu jalankan
`extract_zip_bytes()` — fungsi publik yang sama persis dipakai UI.

In [ ]:
zip_buffer = io.BytesIO()
with zipfile.ZipFile(zip_buffer, "w") as zf:
    for path in image_files:
        zf.write(path, arcname=path.name)
zip_bytes = zip_buffer.getvalue()
print(f"Zip in-memory dibuat: {len(zip_bytes) / 1e6:.1f} MB, {len(image_files)} file")

extract_start = time.perf_counter()
extracted_df = extract_zip_bytes(zip_bytes)
extract_elapsed = time.perf_counter() - extract_start
print(f"Ekstraksi {len(extracted_df)} kwitansi selesai dalam {extract_elapsed:.1f}s "
      f"(rata-rata {extract_elapsed/len(extracted_df):.2f}s/kwitansi)")
extracted_df.drop(columns=["raw_text"])

## 5. Validasi Silang: `extract_uploaded_files()` (sample kecil)

Bukan bagian dari evaluasi akurasi (itu bagian 6 di bawah) — ini cuma
regression check bahwa 2 entry point publik (`extract_zip_bytes` dipakai
halaman upload ZIP, `extract_uploaded_files` dipakai halaman upload foto
individual) tetap menghasilkan output identik untuk file yang sama, karena
keduanya memanggil `_extract_paths()` internal yang sama persis.

In [ ]:
class _FakeUploadedFile:
    """Shim minimal biar path lokal bisa dipakai extract_uploaded_files()
    (yang aslinya menerima objek dari st.file_uploader - punya atribut
    .name & method .getvalue())."""
    def __init__(self, path: Path):
        self.name = path.name
        self._path = path

    def getvalue(self) -> bytes:
        return self._path.read_bytes()


SAMPLE_N = 5
sample_paths = image_files[:SAMPLE_N]
sample_uploaded = [_FakeUploadedFile(p) for p in sample_paths]

sample_via_uploaded = extract_uploaded_files(sample_uploaded).sort_values("source_file").reset_index(drop=True)
sample_via_zip = (
    extracted_df[extracted_df["source_file"].isin([p.name for p in sample_paths])]
    .sort_values("source_file").reset_index(drop=True)
)

compare_cols = [c for c in DETAIL_COLUMNS if c != "raw_text"]
identical = sample_via_uploaded[compare_cols].equals(sample_via_zip[compare_cols])
print(f"extract_uploaded_files() vs extract_zip_bytes() identik di {SAMPLE_N} sample? {identical}")
if not identical:
    diff_mask = (sample_via_uploaded[compare_cols] != sample_via_zip[compare_cols]).any(axis=1)
    print("Baris yang beda (extract_uploaded_files vs extract_zip_bytes):")
    display(sample_via_uploaded.loc[diff_mask, compare_cols])
    display(sample_via_zip.loc[diff_mask, compare_cols])

## 6. Field-Level Accuracy vs Ground Truth

Normalisasi ringan sebelum dibandingkan (whitespace di-trim, huruf
disamakan ke lowercase, `total` dipaksa ke integer) — bukan buat
"melonggarkan" hasil, tapi supaya perbedaan kosmetik yang tidak substantif
(mis. `PENJUALAN` vs `penjualan`) tidak dihitung sebagai kegagalan
ekstraksi, konsisten dengan bagaimana field-field ini sebenarnya dipakai
di sistem (case-insensitive di banyak tempat).

In [ ]:
FIELD_MAP = {
    "jenis_kwitansi": "jenis_kwitansi",
    "no_kwitansi": "no_kwitansi",
    "tanggal": "tanggal",
    "pihak_terkait": "pihak_lain",
    "nama_usaha": "nama_usaha",
    "nik_pemilik": "nik",
    "total": "total_amount",
}

# 4 dari 7 kolom (nama_usaha/tanggal/no_kwitansi/jenis_kwitansi) punya nama
# PERSIS SAMA di extracted_df & ground_truth - kalau langsung di-merge,
# pandas otomatis kasih suffix _pred/_gt dan bikin referensi FIELD_MAP di
# bawah salah ambil kolom. Rename eksplisit ke gt_<field> dulu supaya tidak
# bergantung pada perilaku suffix pandas.
gt_renamed = ground_truth.rename(columns={gt_col: f"gt_{pred_col}" for pred_col, gt_col in FIELD_MAP.items()})
gt_cols_for_merge = ["file_name"] + [f"gt_{pred_col}" for pred_col in FIELD_MAP]
FIELD_MAP_GT = {pred_col: f"gt_{pred_col}" for pred_col in FIELD_MAP}

merged = extracted_df.merge(
    gt_renamed[gt_cols_for_merge], left_on="source_file", right_on="file_name", how="inner",
)
print(f"Baris berhasil di-merge (source_file cocok dgn file_name): {len(merged)}/{len(extracted_df)}")
assert len(merged) == len(extracted_df), "Ada source_file hasil ekstraksi yang tidak ketemu di ground truth - cek nama file."


def _norm(v):
    if pd.isna(v):
        return None
    if isinstance(v, str):
        return " ".join(v.strip().split()).lower()
    return v


def _norm_total(v):
    if pd.isna(v):
        return None
    try:
        return int(float(v))
    except (TypeError, ValueError):
        return None


match_cols = {}
for pred_col, gt_col in FIELD_MAP_GT.items():
    if pred_col == "total":
        pred_norm = merged[pred_col].apply(_norm_total)
        gt_norm = merged[gt_col].apply(_norm_total)
    else:
        pred_norm = merged[pred_col].apply(_norm)
        gt_norm = merged[gt_col].apply(_norm)
    match_cols[pred_col] = (pred_norm == gt_norm) & pred_norm.notna()

match_df = pd.DataFrame(match_cols)
field_accuracy = match_df.mean().sort_values(ascending=False)

print("=== Field-level exact-match rate ===")
for field, rate in field_accuracy.items():
    print(f"  {field:<16}: {rate*100:5.1f}%  ({int(match_df[field].sum())}/{len(match_df)})")

row_success = match_df.all(axis=1)
print(f"\n=== Row-success rate (semua {len(FIELD_MAP)} field benar sekaligus) ===")
print(f"  {row_success.mean()*100:.1f}%  ({int(row_success.sum())}/{len(row_success)})")

### Detail Mismatch (untuk debugging regex `_parse_receipt_text()`)

1 baris per (kwitansi, field) yang gagal — bandingkan `predicted` vs
`ground_truth` langsung, tanpa perlu scroll tabel lebar.

In [ ]:
mismatch_rows = []
for idx, row in merged.iterrows():
    for pred_col, gt_col in FIELD_MAP_GT.items():
        if not match_df.loc[idx, pred_col]:
            mismatch_rows.append({
                "source_file": row["source_file"], "field": pred_col,
                "predicted": row[pred_col], "ground_truth": row[gt_col],
            })
mismatch_df = pd.DataFrame(mismatch_rows)
print(f"Total mismatch (baris x field): {len(mismatch_df)}")
mismatch_df

In [ ]:
# Export raw_text OCR mentah utk kwitansi yang gagal di >=1 field - dipakai
# cek manual apakah masalahnya di model OCR-nya (transkripsi salah baca)
# atau di regex _parse_receipt_text() yang meleset dari format transkripsi.
failing_files = set(mismatch_df["source_file"]) if len(mismatch_df) else set()
failing_detail_df = extracted_df[extracted_df["source_file"].isin(failing_files)]

if len(failing_detail_df):
    raw_text_export = build_raw_text_export(failing_detail_df)
    with open("raw_text_kwitansi_gagal.txt", "w", encoding="utf-8") as f:
        f.write(raw_text_export)
    print(f"Teks OCR mentah utk {len(failing_detail_df)} kwitansi yang gagal disimpan ke raw_text_kwitansi_gagal.txt")
else:
    print("Tidak ada kwitansi yang gagal - semua field cocok dgn ground truth.")

## 7. Dampak ke `compute_monthly_estimates()`

Dibandingkan pakai FUNGSI PRODUKSI YANG SAMA (`compute_monthly_estimates()`
dari `utils/kwitansi_extractor.py`) di 2 input berbeda: (a) hasil OCR
(`extracted_df`), (b) ground truth yang direshape ke skema kolom yang sama
(`total_amount`→`total`, `nik`→`nik_pemilik`, `jenis_kwitansi` di-lowercase)
— supaya perbandingannya murni mengisolasi dampak OCR, bukan perbedaan
logic hitung antara 2 implementasi terpisah.

In [ ]:
gt_for_estimate = ground_truth.rename(columns={"total_amount": "total", "nik": "nik_pemilik"}).copy()
gt_for_estimate["jenis_kwitansi"] = gt_for_estimate["jenis_kwitansi"].str.lower()

true_estimate = compute_monthly_estimates(gt_for_estimate)
ocr_estimate = compute_monthly_estimates(extracted_df)

print("=== Ground truth (dari total_amount asli) ===")
print(true_estimate)
print("\n=== Hasil OCR (dari extract_zip_bytes()) ===")
print(ocr_estimate)

if true_estimate["monthly_turnover_est"] and ocr_estimate["monthly_turnover_est"]:
    diff_pct = (
        (ocr_estimate["monthly_turnover_est"] - true_estimate["monthly_turnover_est"])
        / true_estimate["monthly_turnover_est"] * 100
    )
    print(f"\nSelisih monthly_turnover_est (OCR vs ground truth): {diff_pct:+.1f}%")
else:
    print("\nSalah satu dari monthly_turnover_est (OCR/ground truth) None - tidak bisa hitung selisih persen.")

if true_estimate["transaction_frequency_monthly"] and ocr_estimate["transaction_frequency_monthly"]:
    diff_freq = ocr_estimate["transaction_frequency_monthly"] - true_estimate["transaction_frequency_monthly"]
    print(f"Selisih transaction_frequency_monthly (OCR vs ground truth): {diff_freq:+d} transaksi/bulan")

## Ringkasan & Langkah Selanjutnya

- Field dengan exact-match rate rendah → buka `raw_text_kwitansi_gagal.txt`
  (Sel Bagian 6) untuk lihat transkripsi OCR mentahnya. Biasanya berarti
  regex `_parse_receipt_text()` (`utils/kwitansi_extractor.py`) perlu
  disetel ulang, BUKAN model OCR-nya yang salah baca — LightOnOCR
  transkripsi teksnya sendiri biasanya sudah akurat, masalah paling sering
  ada di pola regex yang meleset dari format transkripsi Markdown/HTML-nya
  (lihat docstring modul soal ini).
- `pihak_terkait` biasanya field yang paling sensitif ke selisih
  whitespace/kapitalisasi minor — exact-match rate rendah di situ tidak
  selalu berarti ekstraksinya salah secara substantif, cek manual dulu di
  `mismatch_df` (Bagian 6) sebelum menyimpulkan.
- Row-success rate adalah metrik paling ketat (representasi realistis dari
  "berapa kwitansi yang bisa dipakai TANPA koreksi manual") — field-level
  accuracy tetap berguna untuk tahu field mana yang paling perlu
  diperbaiki lebih dulu.
- Kalau field accuracy sudah bagus tapi `compute_monthly_estimates()`
  (Bagian 7) masih selisih signifikan dari ground truth, cek apakah baris
  yang gagal di field `total`/`tanggal`/`jenis_kwitansi` (3 field yang
  dipakai fungsi itu) kebetulan terkonsentrasi di bulan/tahun tertentu
  (bias sistematis, bukan random noise) — lihat kolom `tahun`/`bulan` di
  `ground_truth` untuk baris yang match dgn `mismatch_df["source_file"]`.